In [1]:
!pip install apache-beam

In [2]:
import apache_beam as beam
import re
import os

In [3]:
def parse_log(line):
    """Parse Apache/Nginx log line and extract IP, endpoint, and status code."""
    pattern = r'(\S+) .+ "(\w+) (\S+) .+" (\d{3})'
    match = re.match(pattern, line)

    if match:
        ip = match.group(1)
        method = match.group(2)
        endpoint = match.group(3)
        status_code = int(match.group(4))

        return {
            'ip': ip,
            'method': method,
            'endpoint': endpoint,
            'status_code': status_code
        }
    return None

In [4]:
def run(cmd):
    print(f'>> {cmd}')
    os.system(cmd)
    print('')

run('mkdir -p data')

>> mkdir -p data



In [5]:
# Create sample log file
run('mkdir -p data')

sample_logs = """192.168.1.100 - - [28/Oct/2025:10:15:30 +0000] "GET /api/users HTTP/1.1" 200 1523
192.168.1.101 - - [28/Oct/2025:10:15:31 +0000] "POST /api/login HTTP/1.1" 200 456
192.168.1.102 - - [28/Oct/2025:10:15:32 +0000] "GET /api/products HTTP/1.1" 404 234
192.168.1.103 - - [28/Oct/2025:10:15:33 +0000] "GET /api/orders HTTP/1.1" 500 1890
192.168.1.100 - - [28/Oct/2025:10:15:34 +0000] "PUT /api/users HTTP/1.1" 200 789
192.168.1.104 - - [28/Oct/2025:10:15:35 +0000] "GET /api/products HTTP/1.1" 200 3456
192.168.1.100 - - [28/Oct/2025:10:15:36 +0000] "GET /api/orders HTTP/1.1" 200 1234
192.168.1.105 - - [28/Oct/2025:10:15:37 +0000] "GET /api/users HTTP/1.1" 404 567
192.168.1.101 - - [28/Oct/2025:10:15:38 +0000] "POST /api/login HTTP/1.1" 500 2341
192.168.1.106 - - [28/Oct/2025:10:15:39 +0000] "GET /api/products HTTP/1.1" 200 4567
192.168.1.100 - - [28/Oct/2025:10:15:40 +0000] "GET /api/users HTTP/1.1" 200 1523
192.168.1.107 - - [28/Oct/2025:10:15:41 +0000] "GET /api/orders HTTP/1.1" 404 123"""

with open('data/access.log', 'w') as f:
    f.write(sample_logs)

print("Sample log file created!\n")

>> mkdir -p data

Sample log file created!



In [6]:
inputs_pattern = 'data/*.log'
outputs_prefix = 'outputs/log_analysis'

# Running locally with DirectRunner
with beam.Pipeline() as pipeline:

    # Read and parse logs
    parsed_logs = (
        pipeline
        # Read lines from log file
        | 'Read log files' >> beam.io.ReadFromText(inputs_pattern)

        # Parse each log line to extract fields
        | 'Parse log lines' >> beam.Map(parse_log)

        # Filter out any lines that couldn't be parsed
        | 'Filter valid logs' >> beam.Filter(lambda x: x is not None)
    )

    # Count requests per endpoint
    endpoint_counts = (
        parsed_logs
        # Extract endpoint and pair with 1
        | 'Extract endpoints' >> beam.Map(lambda log: (log['endpoint'], 1))

        # Group by endpoint and sum the counts
        | 'Count by endpoint' >> beam.CombinePerKey(sum)

        # Format results for output
        | 'Format endpoint results' >> beam.Map(lambda kv: f"{kv[0]}: {kv[1]} requests")

        # Write to file
        | 'Write endpoint results' >> beam.io.WriteToText(f'{outputs_prefix}_endpoints')
    )

    # Count error requests (status >= 400) per IP
    error_counts = (
        parsed_logs
        # Filter only error responses (4xx and 5xx)
        | 'Filter errors' >> beam.Filter(lambda log: log['status_code'] >= 400)

        # Extract IP address and pair with 1
        | 'Extract error IPs' >> beam.Map(lambda log: (log['ip'], 1))

        # Group by IP and sum error counts
        | 'Count errors by IP' >> beam.CombinePerKey(sum)

        # Format results for output
        | 'Format error results' >> beam.Map(lambda kv: f"{kv[0]}: {kv[1]} error")

        # Write to file
        | 'Write error results' >> beam.io.WriteToText(f'{outputs_prefix}_errors')
    )

print("Pipeline completed!\n")

Pipeline completed!

